In [3]:
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")


In [4]:
from trainedModel import predict_next_6_months

In [5]:
import pandas as pd
import json

# Assuming 'predict_next_6_months' is defined and working properly

df = pd.read_csv("crop_prices.csv")

# Define the prediction function
def forecast_to_json(crop, market, df):
    forecast = predict_next_6_months(crop, market, df)

    # Check if forecast is available
    if forecast:
        # Convert forecast dictionary to a JSON-compatible format, ensuring float32 is converted to standard float
        forecast_json = {str(date): float(price) for date, price in forecast.items()}
        return json.dumps(forecast_json, indent=2)
    else:
        return json.dumps({"error": "No forecast available for the given crop and market."}, indent=2)

# Example usage
result_json = forecast_to_json("Rice", "Sealdah Koley Market", df)
print(result_json)




{
  "2025-07-31": 3092.159912109375,
  "2025-08-31": 2977.7099609375,
  "2025-09-30": 2890.6201171875,
  "2025-10-31": 3000.110107421875,
  "2025-11-30": 3052.139892578125,
  "2025-12-31": 2939.030029296875
}


In [6]:
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
import os

In [7]:
from trainedModel import crop_profit_recommendation_with_risk

In [8]:
import pandas as pd
import pickle
import warnings
import numpy as np
warnings.filterwarnings("ignore", message="X does not have valid feature names")


# Load your dataset
df = pd.read_csv("crop_prices.csv")
with open('crop_recommendation_model.pkl', 'rb') as f:
    pipeline = pickle.load(f)
crops = []

# Extract components
model = pipeline['model']
scaler = pipeline['scaler']
label_encoder = pipeline['label_encoder']
feature_columns = pipeline['feature_columns']

# New sample input (N, P, K, Temperature, Humidity, Ph, Rain)
new_sample = [90, 42, 43, 20.87, 82.00, 6.5, 200.0]
new_sample_df = pd.DataFrame([new_sample], columns=feature_columns)
new_sample_scaled = scaler.transform(new_sample_df)

# Predict probabilities
pred_proba = model.predict_proba(new_sample_scaled)
top_5_indices = np.argsort(pred_proba[0])[::-1][:5]
top_5_crops = label_encoder.inverse_transform(top_5_indices)

print("🌾 Top 5 Recommended Crops:")
for i, crop in enumerate(top_5_crops, 1):
    print(f"{i}. {crop}")

# Assign predicted crops for market recommendation
crops = top_5_crops.tolist()


# Define crops


# Get all unique markets from Kolkata
KolkataMarkets = df["Market"].unique().tolist()

# Load the model pipeline (make sure this file exists and is trained)
with open(r'crop_recommendation_model.pkl', 'rb') as f:
    model_pipeline = pickle.load(f)

# Collect recommendations
recommendations = []

for market in KolkataMarkets:
    try:
        result = crop_profit_recommendation_with_risk(df, crops, market)
        if result:
            recommendations.extend(result)  # assuming result is a list of dicts
    except Exception as e:
        print(f"Skipping market {market} due to error: {e}")

# Convert to DataFrame if needed
recommendations_df = pd.DataFrame(recommendations)

# Print or save results
print(recommendations_df.to_json(orient='records', indent=2))


🌾 Top 5 Recommended Crops:
1. Banana
2. Rice
3. Bhindi(Ladies Finger)
4. Apple
5. Grapes
[
  {
    "Crop":"Apple",
    "Avg_Predicted_Price (\u20b9\/qtl)":14808.1201171875,
    "Net_Profit_per_Acre (\u20b9)":13656.82,
    "Risk_Factor":"\ud83d\udd34 High",
    "Monthly_Prices":{
      "2025-06-30":13380.4501953125,
      "2025-07-31":14368.150390625,
      "2025-08-31":14783.6396484375,
      "2025-09-30":15825.98046875,
      "2025-10-31":15747.8896484375,
      "2025-11-30":14742.58984375
    },
    "message":null
  },
  {
    "Crop":"Banana",
    "Avg_Predicted_Price (\u20b9\/qtl)":1620.6700439453,
    "Net_Profit_per_Acre (\u20b9)":6723.57,
    "Risk_Factor":"\ud83d\udd34 High",
    "Monthly_Prices":{
      "2024-07-31":1546.1999511719,
      "2024-08-31":1669.5799560547,
      "2024-09-30":1610.1199951172,
      "2024-10-31":1635.6600341797,
      "2024-11-30":1617.8000488281,
      "2024-12-31":1644.6999511719
    },
    "message":null
  },
  {
    "Crop":null,
    "Avg_Predicted

In [13]:
# input (N, P, K, Temperature, Humidity, Ph, Rain)
new_sample = [50, 50, 60, 80.87, 50.00, 7, 500.0]
from cropRecommend import cropProfit


cropProfit(new_sample)

Rice


['Rice']